# Ray RLlib: Custom playground + PPO (`TicketQueue-v0`)

Project path: `projects/custom-env-ppo`

Define a **custom Gymnasium env**, `register_env` it with Ray, and train **PPO** — companion step 6.

**Setup (once)** from the repository root:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r projects/custom-env-ppo/requirements.txt
```

Select that kernel. Script twin: `python projects/custom-env-ppo/train_queue_ppo.py`.

**How to match a use case:** see [README — How to define a playground for your use case](README.md#how-to-define-a-playground-for-your-use-case) (obs / action / reward / episode mapping). Edit [`queue_env.py`](queue_env.py) to change costs, arrivals, or observations — then re-run.

## 1. Inspect the playground

Roll out a few random steps to see queue length, rewards, and overflow events.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "queue_env.py").exists():
    PROJECT_DIR = Path.cwd() / "projects" / "custom-env-ppo"
sys.path.insert(0, str(PROJECT_DIR.resolve()))

from queue_env import TicketQueueEnv

env = TicketQueueEnv()
obs, info = env.reset(seed=0)
print("reset obs", obs, "info", info)
total = 0.0
for t in range(8):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total += float(reward)
    print(
        f"t={t} action={action} reward={reward:.2f} "
        f"queue={info['queue_length']} overflow={info['overflow']}"
    )
    if terminated or truncated:
        break
print("partial return", round(total, 2))
env.close()

## 2. Register env + train PPO

In [ ]:
import warnings
from typing import Any

warnings.filterwarnings(
    "ignore",
    message=r".*RLModule\(config=\[RLModuleConfig object\]\).*",
    category=DeprecationWarning,
)

from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig
from ray.tune.registry import register_env

from queue_env import ENV_NAME, make_ticket_queue_env


def episode_return_mean(result: dict[str, Any]) -> float | None:
    env_runners = result.get("env_runners") or {}
    value = env_runners.get("episode_return_mean")
    return float(value) if value is not None else None


register_env(ENV_NAME, make_ticket_queue_env)

config = (
    PPOConfig()
    .environment(ENV_NAME)
    .env_runners(num_env_runners=2)
    .rl_module(model_config=DefaultModelConfig(fcnet_hiddens=[64, 64]))
    .evaluation(evaluation_num_env_runners=1)
    .debugging(log_level="ERROR")
)

algo = config.build_algo()
try:
    for i in range(1, 13):
        result = algo.train()
        ret = episode_return_mean(result)
        steps = result.get("num_env_steps_sampled_lifetime")
        if ret is not None:
            print(f"iter={i}  episode_return_mean={ret:.1f}  env_steps={steps}")
        else:
            print(f"iter={i}  env_steps={steps}")

    eval_result = algo.evaluate()
    eval_ret = episode_return_mean(eval_result)
    print(
        f"evaluate  episode_return_mean={eval_ret:.1f}"
        if eval_ret is not None
        else "evaluate  (no episode_return_mean)"
    )
finally:
    algo.stop()

## What this teaches

| Idea | In this notebook |
| --- | --- |
| Custom playground | `gymnasium.Env` with your rewards and dynamics |
| Ray registration | `register_env(name, factory)` |
| Same trainer as Taxi | `PPOConfig` → `build_algo()` → `train` / `evaluate` / `stop` |

Next ideas: change `queue_cost` / `arrival_prob`, add action masking, or record Parquet for [offline BC](../offline-marwil/offline_bc.ipynb). · [Project README](README.md)